# Evaluation Metrics

This notebook defines the evaluation functions used by every baseline notebook and the hybrid fusion notebook. After the functions are defined, the notebook **writes them out as `recsys_metrics.py` on Google Drive** so downstream notebooks can simply `import` them rather than re-defining everything.

## What's measured

Two flavors of metric, on the same set of test interactions:

**Pointwise rating prediction:**
- **RMSE** — root mean squared error of predicted vs true ratings.
- **MAE** — mean absolute error.

**Top-N ranking quality (the corrected protocol):**
- **HR@10** — hit ratio at 10. For each test interaction, the true item is mixed with 99 sampled unseen items; the model ranks the 100 candidates; HR@10 = 1 if the true item is in the top 10, else 0.
- **NDCG@10** — same setup, scored by `1 / log2(rank + 1)` if the true item is in the top 10 else 0.
- **Precision@10, Recall@10, F1@10** — relevant = test rating ≥ 3.5; computed per user from the model's top-10 ranking among test+sampled candidates.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
import os, random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from scipy.stats import wilcoxon

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

ARTIFACT_DIR = Path('/content/gdrive/MyDrive/recsys_artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## Pointwise metrics: RMSE, MAE

In [3]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))

## Sampled-negatives candidate generator

Given the training set, build a per-user set of *seen* items. For each test interaction we sample `num_negatives` items the user has not seen.

This is deterministic given the seed — important for reproducibility and for fair comparison across methods.

In [4]:
def build_candidate_lists(train_df, test_df, num_negatives=99, seed=42):
    """
    For each test interaction, build [true_item, neg_1, ..., neg_K] candidate list.

    Returns:
        candidates: dict mapping (userId, true_movieId) -> np.array of K+1 movieIds,
                    with the true movieId at index 0.
    """
    rng = np.random.RandomState(seed)
    all_items = np.array(sorted(set(train_df['movieId']) | set(test_df['movieId'])))
    seen_by_user = train_df.groupby('userId')['movieId'].apply(set).to_dict()

    candidates = {}
    for _, row in test_df.iterrows():
        user, true_item = int(row['userId']), int(row['movieId'])
        seen = seen_by_user.get(user, set()) | {true_item}
        unseen_mask = ~np.isin(all_items, list(seen))
        unseen = all_items[unseen_mask]
        if len(unseen) < num_negatives:
            # Fall back: sample with replacement if the user has seen almost everything
            negs = rng.choice(unseen, size=num_negatives, replace=True)
        else:
            negs = rng.choice(unseen, size=num_negatives, replace=False)
        candidates[(user, true_item)] = np.concatenate([[true_item], negs])
    return candidates

## Ranking metrics: HR@k and NDCG@k (sampled negatives)

Each method exposes a `predict_fn(user_id, movie_ids) -> np.ndarray` that returns a score per candidate. Higher score = more preferred. The metric functions below rank the candidates by that score and check the position of the true item.

In [5]:
def hr_ndcg_at_k(predict_fn, test_df, candidates, k=10):
    """
    Compute HR@k and NDCG@k using sampled negatives.

    Args:
        predict_fn: callable (user_id: int, movie_ids: np.ndarray) -> np.ndarray of scores
        test_df: DataFrame with columns [userId, movieId, rating]
        candidates: dict (user, true_item) -> np.array of 1 + N_neg movieIds
        k: cutoff

    Returns:
        dict with mean 'hr_at_k', 'ndcg_at_k', and per-user dicts for significance tests.
    """
    hits, ndcgs = [], []
    per_user_hr, per_user_ndcg = {}, {}

    for _, row in test_df.iterrows():
        user, true_item = int(row['userId']), int(row['movieId'])
        cands = candidates[(user, true_item)]
        scores = np.asarray(predict_fn(user, cands), dtype=float)

        # Rank descending; resolve ties by index (deterministic).
        order = np.argsort(-scores, kind='stable')
        ranked_items = cands[order]

        # Position of the true item (the true item is at index 0 in `cands`).
        true_position = int(np.where(ranked_items == true_item)[0][0])  # 0-indexed

        hit  = 1.0 if true_position < k else 0.0
        ndcg = (1.0 / np.log2(true_position + 2)) if true_position < k else 0.0

        hits.append(hit)
        ndcgs.append(ndcg)
        per_user_hr[user]   = hit
        per_user_ndcg[user] = ndcg

    return {
        'hr_at_k':   float(np.mean(hits)),
        'ndcg_at_k': float(np.mean(ndcgs)),
        'per_user_hr':   per_user_hr,
        'per_user_ndcg': per_user_ndcg,
    }

## Precision / Recall / F1 @ k

Per user: rank the candidate set (true item + sampled negatives) by predicted score, take top-k. An item is *relevant* if it's the true held-out test item AND the user's true rating on it is ≥ `threshold`.

With sampled-negatives + LOO, each user has exactly one relevant item per test interaction, so precision and recall behave a bit like HR — but with a threshold filter on rating. We report them mainly for continuity with the original report's metric set.

In [6]:
def precision_recall_f1_at_k(predict_fn, test_df, candidates, k=10, threshold=3.5):
    precs, recs = [], []
    for _, row in test_df.iterrows():
        user, true_item, true_rating = int(row['userId']), int(row['movieId']), float(row['rating'])
        is_relevant = true_rating >= threshold
        cands = candidates[(user, true_item)]
        scores = np.asarray(predict_fn(user, cands), dtype=float)
        order = np.argsort(-scores, kind='stable')
        top_k_items = cands[order][:k]

        hit_in_top_k = (true_item in top_k_items) and is_relevant
        # In this LOO + sampled-negatives setup there is at most one relevant item per query.
        precs.append((1.0 / k) if hit_in_top_k else 0.0)
        recs.append(1.0 if hit_in_top_k else 0.0)

    precision = float(np.mean(precs))
    recall    = float(np.mean(recs))
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return {'precision_at_k': precision, 'recall_at_k': recall, 'f1_at_k': f1}

## End-to-end evaluator

`evaluate_model` is the single entry point every baseline notebook calls. It expects:

- `predict_fn`: callable `(user_id, movie_ids) -> scores` for the ranking metrics.
- `pointwise_predictions`: a DataFrame `[userId, movieId, rating, predicted_rating]` for RMSE/MAE. For models that produce ratings on a 1–5 scale (SVD, KNN, etc.), this is the per-test-row predicted rating. For models that only produce ranking scores (e.g., a pure embedding similarity), pass `None` to skip RMSE/MAE.
- `train_df`, `test_df`, `candidates`: as built above.

Returns a flat dict of all metrics and per-user diagnostics.

In [7]:
def evaluate_model(predict_fn, test_df, candidates, *, pointwise_predictions=None, k=10, threshold=3.5):
    out = {}
    if pointwise_predictions is not None:
        y_true = pointwise_predictions['rating'].values
        y_pred = pointwise_predictions['predicted_rating'].values
        out['rmse'] = rmse(y_true, y_pred)
        out['mae']  = mae(y_true, y_pred)

    ranking = hr_ndcg_at_k(predict_fn, test_df, candidates, k=k)
    out['hr_at_k']        = ranking['hr_at_k']
    out['ndcg_at_k']      = ranking['ndcg_at_k']
    out['per_user_hr']    = ranking['per_user_hr']
    out['per_user_ndcg']  = ranking['per_user_ndcg']

    prf = precision_recall_f1_at_k(predict_fn, test_df, candidates, k=k, threshold=threshold)
    out.update(prf)

    return out

## Significance testing (paired Wilcoxon)

For two methods A and B, compare per-user NDCG@10 (or any other per-user metric) with a paired Wilcoxon signed-rank test. Used in the results table to mark which differences between methods are statistically significant.

In [8]:
def paired_wilcoxon(per_user_a, per_user_b):
    """Returns (statistic, p_value) for paired per-user metrics. Users missing from either side are dropped."""
    common = sorted(set(per_user_a) & set(per_user_b))
    a = np.array([per_user_a[u] for u in common])
    b = np.array([per_user_b[u] for u in common])
    if np.allclose(a, b):
        return 0.0, 1.0
    stat, p = wilcoxon(a, b, zero_method='wilcox', alternative='two-sided')
    return float(stat), float(p)

## Smoke test

Synthetic 5-user, 50-item dataset to confirm the metric implementations behave sensibly.

In [9]:
rng = np.random.RandomState(0)
n_users, n_items = 5, 50
train_synth = pd.DataFrame({
    'userId':  np.repeat(np.arange(n_users), 8),
    'movieId': rng.choice(n_items, size=n_users * 8, replace=True),
    'rating':  rng.uniform(1, 5, size=n_users * 8),
}).drop_duplicates(['userId', 'movieId'])

test_synth = pd.DataFrame({
    'userId':  np.arange(n_users),
    'movieId': [99, 98, 97, 96, 95],
    'rating':  [5.0, 4.5, 1.0, 4.0, 3.0],
})
cands_synth = build_candidate_lists(train_synth, test_synth, num_negatives=20, seed=42)

# Oracle: always scores the true item highest. Should give HR=NDCG=1.
def oracle(u, items):
    s = np.zeros(len(items))
    true_item = test_synth.set_index('userId').loc[u, 'movieId']
    s[items == true_item] = 100
    return s

# Random scorer: should give chance-level metrics.
def random_scorer(u, items):
    return np.random.RandomState(int(u)).rand(len(items))

print('Oracle:', hr_ndcg_at_k(oracle, test_synth, cands_synth, k=10))
print('Random:', hr_ndcg_at_k(random_scorer, test_synth, cands_synth, k=10))

Oracle: {'hr_at_k': 1.0, 'ndcg_at_k': 1.0, 'per_user_hr': {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0}, 'per_user_ndcg': {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0), 4: np.float64(1.0)}}
Random: {'hr_at_k': 0.6, 'ndcg_at_k': 0.20704125223540193, 'per_user_hr': {0: 0.0, 1: 1.0, 2: 0.0, 3: 1.0, 4: 1.0}, 'per_user_ndcg': {0: 0.0, 1: np.float64(0.2890648263178879), 2: 0.0, 3: np.float64(0.31546487678572877), 4: np.float64(0.43067655807339306)}}


## Export as `recsys_metrics.py`

Every baseline notebook will `sys.path.append(ARTIFACT_DIR)` and `from recsys_metrics import evaluate_model, build_candidate_lists, ...` so we don't redefine these functions five times.

In [10]:
module_source = '''\
"""
Evaluation utilities for the recommender baselines + hybrid fusion.

Defined in evaluation_metrics.ipynb; exported here so notebooks can import.
"""
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def build_candidate_lists(train_df, test_df, num_negatives=99, seed=42):
    rng = np.random.RandomState(seed)
    all_items = np.array(sorted(set(train_df["movieId"]) | set(test_df["movieId"])))
    seen_by_user = train_df.groupby("userId")["movieId"].apply(set).to_dict()
    candidates = {}
    for _, row in test_df.iterrows():
        user, true_item = int(row["userId"]), int(row["movieId"])
        seen = seen_by_user.get(user, set()) | {true_item}
        unseen = all_items[~np.isin(all_items, list(seen))]
        replace = len(unseen) < num_negatives
        negs = rng.choice(unseen, size=num_negatives, replace=replace)
        candidates[(user, true_item)] = np.concatenate([[true_item], negs])
    return candidates


def hr_ndcg_at_k(predict_fn, test_df, candidates, k=10):
    hits, ndcgs = [], []
    per_user_hr, per_user_ndcg = {}, {}
    for _, row in test_df.iterrows():
        user, true_item = int(row["userId"]), int(row["movieId"])
        cands = candidates[(user, true_item)]
        scores = np.asarray(predict_fn(user, cands), dtype=float)
        order = np.argsort(-scores, kind="stable")
        ranked = cands[order]
        pos = int(np.where(ranked == true_item)[0][0])
        hit = 1.0 if pos < k else 0.0
        ndcg = (1.0 / np.log2(pos + 2)) if pos < k else 0.0
        hits.append(hit); ndcgs.append(ndcg)
        per_user_hr[user] = hit
        per_user_ndcg[user] = ndcg
    return {
        "hr_at_k": float(np.mean(hits)),
        "ndcg_at_k": float(np.mean(ndcgs)),
        "per_user_hr": per_user_hr,
        "per_user_ndcg": per_user_ndcg,
    }


def precision_recall_f1_at_k(predict_fn, test_df, candidates, k=10, threshold=3.5):
    precs, recs = [], []
    for _, row in test_df.iterrows():
        user, true_item, true_rating = int(row["userId"]), int(row["movieId"]), float(row["rating"])
        is_relevant = true_rating >= threshold
        cands = candidates[(user, true_item)]
        scores = np.asarray(predict_fn(user, cands), dtype=float)
        order = np.argsort(-scores, kind="stable")
        top_k = cands[order][:k]
        hit = (true_item in top_k) and is_relevant
        precs.append((1.0 / k) if hit else 0.0)
        recs.append(1.0 if hit else 0.0)
    precision = float(np.mean(precs))
    recall = float(np.mean(recs))
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return {"precision_at_k": precision, "recall_at_k": recall, "f1_at_k": f1}


def evaluate_model(predict_fn, test_df, candidates, *, pointwise_predictions=None, k=10, threshold=3.5):
    out = {}
    if pointwise_predictions is not None:
        out["rmse"] = rmse(pointwise_predictions["rating"], pointwise_predictions["predicted_rating"])
        out["mae"]  = mae(pointwise_predictions["rating"], pointwise_predictions["predicted_rating"])
    ranking = hr_ndcg_at_k(predict_fn, test_df, candidates, k=k)
    out["hr_at_k"] = ranking["hr_at_k"]
    out["ndcg_at_k"] = ranking["ndcg_at_k"]
    out["per_user_hr"] = ranking["per_user_hr"]
    out["per_user_ndcg"] = ranking["per_user_ndcg"]
    out.update(precision_recall_f1_at_k(predict_fn, test_df, candidates, k=k, threshold=threshold))
    return out


def paired_wilcoxon(per_user_a, per_user_b):
    common = sorted(set(per_user_a) & set(per_user_b))
    a = np.array([per_user_a[u] for u in common])
    b = np.array([per_user_b[u] for u in common])
    if np.allclose(a, b):
        return 0.0, 1.0
    stat, p = wilcoxon(a, b, zero_method="wilcox", alternative="two-sided")
    return float(stat), float(p)
'''

out_path = ARTIFACT_DIR / 'recsys_metrics.py'
out_path.write_text(module_source)
print(f'Wrote {out_path}')
print('Downstream notebooks should:')
print('    import sys; sys.path.append("/content/gdrive/MyDrive/recsys_artifacts")')
print('    from recsys_metrics import evaluate_model, build_candidate_lists, paired_wilcoxon')

Wrote /content/gdrive/MyDrive/recsys_artifacts/recsys_metrics.py
Downstream notebooks should:
    import sys; sys.path.append("/content/gdrive/MyDrive/recsys_artifacts")
    from recsys_metrics import evaluate_model, build_candidate_lists, paired_wilcoxon
